# SEED BFCL OPD-only: June 24 fixed-4 A100 smoke

This notebook uses only the historical June 24 three-field Skill-SD summaries. It runs a privileged arm and a same-prompt control from the same pinned base model.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
import torch
gpu_names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
assert len(gpu_names) == 1 and 'A100' in gpu_names[0].upper(), (torch.cuda.device_count(), gpu_names)
print('SEED_BFCL_A100_OK', gpu_names[0])

In [ ]:
import os, pathlib, subprocess, sys
SEED_URL = 'https://github.com/Alexishiyu/SEED.git'
SEED_BRANCH = 'codex/bfcl-opsd'
SEED_ROOT = pathlib.Path('/content/SEED')
GORILLA_ROOT = pathlib.Path('/content/gorilla')
BFCL_ROOT = GORILLA_ROOT / 'berkeley-function-call-leaderboard'
BFCL_COMMIT = 'f7cf7359b7ac615a0b294831c5ba2bc95ee4a000'
def run(cmd, cwd=None):
    print('RUN', ' '.join(map(str, cmd)), flush=True)
    subprocess.run(list(map(str, cmd)), cwd=cwd, check=True)
if not (SEED_ROOT / '.git').is_dir():
    run(['git', 'clone', '--branch', SEED_BRANCH, SEED_URL, SEED_ROOT])
run(['git', 'fetch', 'origin', SEED_BRANCH], cwd=SEED_ROOT)
seed_commit = subprocess.check_output(['git', 'rev-parse', 'FETCH_HEAD'], cwd=SEED_ROOT, text=True).strip()
run(['git', 'checkout', '--detach', seed_commit], cwd=SEED_ROOT)
if not (GORILLA_ROOT / '.git').is_dir():
    run(['git', 'clone', 'https://github.com/ShishirPatil/gorilla.git', GORILLA_ROOT])
run(['git', 'fetch', 'origin', BFCL_COMMIT], cwd=GORILLA_ROOT)
run(['git', 'checkout', '--detach', BFCL_COMMIT], cwd=GORILLA_ROOT)
run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip', 'setuptools', 'wheel'])
run([sys.executable, '-m', 'pip', 'install', '-q', 'vllm==0.11.0', 'peft==0.17.1', 'pandas', 'pyarrow'])
run([sys.executable, '-m', 'pip', 'install', '-q', 'https://github.com/Dao-AILab/flash-attention/releases/download/v2.8.3/flash_attn-2.8.3%2Bcu12torch2.8cxx11abiTRUE-cp312-cp312-linux_x86_64.whl'])
run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(SEED_ROOT)])
run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(BFCL_ROOT)])
run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'vllm==0.11.0'])
print('SEED_BFCL_SETUP_OK seed_sha=' + seed_commit + ' bfcl_sha=' + BFCL_COMMIT)

In [ ]:
test_files = [
    'tests/trainer/ppo/test_opd_loss.py',
    'tests/trainer/ppo/test_seed_advantage.py',
    'tests/trainer/ppo/test_seed_analyzer.py',
    'tests/trainer/ppo/test_episode_skill_guidance.py',
    'tests/trainer/ppo/test_seed_skill_gen_reward.py',
    'tests/seed/test_june24_skill_summary.py',
    'tests/environments/test_bfcl_env.py',
    'tests/trainer/ppo/test_opd_only_objective.py',
]
run([sys.executable, '-m', 'pytest', '-q', *test_files], cwd=SEED_ROOT)
print('SEED_BFCL_TESTS_OK')

In [ ]:
from datetime import datetime, timezone
import json, shutil
sys.path.insert(0, str(SEED_ROOT))
from seed.june24_skill_summary import build_fixed_manifest, materialize_skill_bank
stamp = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
RUN_ROOT = pathlib.Path('/content/drive/MyDrive/bfcl_qwen_experiment/seed_opsd_colab') / f'june24_skill_summary_smoke_{stamp}'
INPUTS = RUN_ROOT / 'inputs'
INPUTS.mkdir(parents=True, exist_ok=False)
SOURCE_DIRS = [
    pathlib.Path('/content/drive/MyDrive/bfcl_qwen_experiment/a100_skill_sd_50_20260624_055240/skills_openai'),
    pathlib.Path('/content/drive/MyDrive/bfcl_qwen_experiment/a100_skill_sd_150_50_199_20260624_063820/skills_openai'),
]
for source in SOURCE_DIRS: assert source.is_dir(), source
canonical_fixed_csv = pathlib.Path('/content/drive/MyDrive/bfcl_qwen_pairwise_analysis/15fRBFq4gbXgJeQ5CHO_bVjeEp0mlH9rB/fixed_tasks.csv')
assert canonical_fixed_csv.is_file(), canonical_fixed_csv
fixed_csv = INPUTS / 'fixed_tasks.csv'
shutil.copy2(canonical_fixed_csv, fixed_csv)
rejected_75 = pathlib.Path('/content/drive/MyDrive/bfcl_qwen_experiment/opsd_verl_a100/june24_teacher_success75_20260718_164930/inputs/teacher_success_selection.json')
assert rejected_75.is_file(), rejected_75
fixed_manifest_path = INPUTS / 'june24_fixed40_manifest.json'
fixed_manifest = build_fixed_manifest(fixed_csv, rejected_75)
fixed_manifest_path.write_text(json.dumps(fixed_manifest, indent=2) + '\n', encoding='utf-8')
task_ids = fixed_manifest['task_ids'][:4]
skill_bank_path = INPUTS / 'june24_fixed4_skill_bank.json'
materialize_skill_bank(source_dirs=SOURCE_DIRS, fixed_manifest=fixed_manifest_path, output_path=skill_bank_path, selected_task_ids=task_ids)
(RUN_ROOT / 'metadata').mkdir(parents=True, exist_ok=True)
(RUN_ROOT / 'metadata' / 'resolved_seed_commit.txt').write_text(seed_commit + '\n', encoding='utf-8')
print('SEED_BFCL_INPUTS_OK run_root=' + str(RUN_ROOT), 'task_ids=' + ','.join(task_ids))

In [ ]:
MODEL = 'Qwen/Qwen3-4B-Instruct-2507'
RLPAPER_SHA = '690946ca3e1d2f257d7c1acdff4df4eb1feed1ec'
launcher = SEED_ROOT / 'examples/seed_trainer/run_bfcl_opsd.py'
common = [sys.executable, str(launcher), '--june24-skill-bank', str(skill_bank_path), '--fixed-manifest', str(fixed_manifest_path), '--task-ids', ','.join(task_ids), '--run-root', str(RUN_ROOT), '--bfcl-root', str(BFCL_ROOT), '--model', MODEL, '--batch-size', '4', '--updates', '1', '--rlpaper-sha', RLPAPER_SHA]
run(common, cwd=SEED_ROOT)
run(common + ['--same-prompt-control'], cwd=SEED_ROOT)
print('SEED_BFCL_PLAN_EXPORT_PREFLIGHT_OK')

In [ ]:
before_root = RUN_ROOT / 'official_bfcl_before'
run([sys.executable, str(SEED_ROOT / 'examples/seed_trainer/evaluate_bfcl_checkpoint.py'), '--bfcl-root', str(BFCL_ROOT), '--project-root', str(before_root), '--model', MODEL, '--task-ids', ','.join(task_ids)], cwd=SEED_ROOT)
print('SEED_BFCL_BEFORE_EVAL_OK')

In [ ]:
run(common + ['--execute'], cwd=SEED_ROOT)
print('SEED_BFCL_PRIVILEGED_TRAIN_OK')

In [ ]:
export_root = RUN_ROOT / 'exports' / 'privileged_june24_merged'
export_evidence = RUN_ROOT / 'evidence' / 'privileged_june24' / 'checkpoint_export.json'
run([sys.executable, str(SEED_ROOT / 'examples/seed_trainer/merge_bfcl_opsd_lora.py'), '--checkpoint-root', str(RUN_ROOT / 'checkpoints' / 'privileged_june24'), '--base-model', MODEL, '--output', str(export_root), '--evidence', str(export_evidence), '--validate-vllm'], cwd=SEED_ROOT)
print('SEED_BFCL_MERGE_RELOAD_OK')

In [ ]:
after_root = RUN_ROOT / 'official_bfcl_after'
run([sys.executable, str(SEED_ROOT / 'examples/seed_trainer/evaluate_bfcl_checkpoint.py'), '--bfcl-root', str(BFCL_ROOT), '--project-root', str(after_root), '--model', MODEL, '--local-model-path', str(export_root), '--task-ids', ','.join(task_ids), '--port', '8001'], cwd=SEED_ROOT)
print('SEED_BFCL_AFTER_EVAL_OK')

In [ ]:
run(common + ['--same-prompt-control', '--execute'], cwd=SEED_ROOT)
print('SEED_BFCL_CONTROL_TRAIN_OK')

In [ ]:
final_report = RUN_ROOT / 'evidence' / 'seed_bfcl_smoke_complete.json'
run([sys.executable, str(SEED_ROOT / 'examples/seed_trainer/collect_bfcl_opsd_evidence.py'), '--run-root', str(RUN_ROOT), '--before-eval', str(before_root / 'official_bfcl_summary.json'), '--after-eval', str(after_root / 'official_bfcl_summary.json'), '--export-evidence', str(export_evidence), '--output', str(final_report)], cwd=SEED_ROOT)
report = json.loads(final_report.read_text(encoding='utf-8'))
assert report['status'] == 'complete'
print('SEED_BFCL_SMOKE_COMPLETE', final_report)
print(json.dumps(report['comparison'], indent=2))